In [ ]:
import pandas as pd
import numpy as np
import ta
from datetime import timedelta
# Assuming df is your DataFrame containing 'open', 'high', 'low', 'close', and 'volume' columns.
# Example: df = pd.read_csv('your_data.csv')
# Assuming df is your DataFrame containing 'open', 'high', 'low', 'close', and 'volume' columns.
# Example: df = pd.read_csv('your_data.csv')
file_path = 'common/MachineLearningModel/output/five_mins/EURUSD_5_Min_testing_new.csv'
df = pd.read_csv(file_path).reset_index(drop=True)
# Parameters
FasterMovingAverage = 5
SlowerMovingAverage = 12
RSIPeriod = 12
MagicFilterPeriod = 1
BollingerbandsPeriod = 10
BollingerbandsShift = 0
BollingerbandsDeviation = 0.5
BullsPowerPeriod = 50
BearsPowerPeriod = 50
Utstup = 10
Alerts = True

# Calculate Indicators
df['ema_fast'] = ta.trend.ema_indicator(df['close'], window=FasterMovingAverage)
df['ema_slow'] = ta.trend.ema_indicator(df['close'], window=SlowerMovingAverage)
df['rsi'] = ta.momentum.rsi(df['close'], window=RSIPeriod)
df['bulls_power'] = df['high'] - ta.trend.ema_indicator(df['close'], window=BullsPowerPeriod)
df['bears_power'] = df['low'] - ta.trend.ema_indicator(df['close'], window=BearsPowerPeriod)
bb = ta.volatility.BollingerBands(df['close'], window=BollingerbandsPeriod, window_dev=BollingerbandsDeviation)
df['bb_upper'] = bb.bollinger_hband()
df['bb_lower'] = bb.bollinger_lband()

# Initialize conditions
Gi_132 = False
Gi_136 = False
Gi_140 = False
Gi_144 = False
Gi_148 = False
Gi_152 = False
Gi_156 = False
Gi_160 = False
Gi_164 = False
Gi_168 = False
Gi_172 = 0
Gi_176 = False
Gi_180 = False

df['signal_buy'] = np.nan
df['signal_sell'] = np.nan

# Loop through data
for i in range(len(df)-1,1,-1):
    if i < 10:
        continue  # Skip initial periods where sufficient data isn't available

    # Calculate Ld_124
    Ld_140 = np.sum(np.abs(df['high'][i:i + 10] - df['low'][i:i + 10]))
    Ld_132 = Ld_140 / 10.0
    Ld_124 = 100 - 100.0 * ((Ld_132 - 0.0) / 10.0)

    if Ld_124 >= 0.0:
        Gi_148 = True
        Gi_168 = False
    else:
        Gi_148 = False
        Gi_168 = True

    if df['close'][i] > df['bb_upper'][i] and df['close'][i - 1] >= df['bb_upper'][i - 1]:
        Gi_144 = False
        Gi_164 = True

    if df['close'][i] < df['bb_lower'][i] and df['close'][i - 1] <= df['bb_lower'][i - 1]:
        Gi_144 = True
        Gi_164 = False

    if df['bulls_power'][i] > 0.0 and df['bulls_power'][i - 1] > df['bulls_power'][i]:
        Gi_140 = False
        Gi_160 = True

    if df['bears_power'][i] < 0.0 and df['bears_power'][i - 1] < df['bears_power'][i]:
        Gi_140 = True
        Gi_160 = False

    if df['rsi'][i] > 50.0 and df['rsi'][i - 1] < 50.0:
        Gi_136 = True
        Gi_156 = False

    if df['rsi'][i] < 50.0 and df['rsi'][i - 1] > 50.0:
        Gi_136 = False
        Gi_156 = True

    if df['ema_fast'][i] > df['ema_slow'][i] and df['ema_fast'][i - 1] < df['ema_slow'][i - 1]:
        Gi_132 = True
        Gi_152 = False

    if df['ema_fast'][i] < df['ema_slow'][i] and df['ema_fast'][i - 1] > df['ema_slow'][i - 1]:
        Gi_132 = False
        Gi_152 = True

    # Check buy conditions
    if (Gi_132 and Gi_136 and Gi_144 and Gi_140 and Gi_148 and Gi_172 != 1):
        df.at[i, 'signal_buy'] = df['low'][i] - (Utstup * df['low'][i])
        Gi_172 = 1

    # Check sell conditions
    elif (Gi_152 and Gi_156 and Gi_164 and Gi_160 and not Gi_168 and Gi_172 != 2):
        df.at[i, 'signal_sell'] = df['high'][i] + (Utstup * df['high'][i])
        Gi_172 = 2

df['UTC'] = pd.to_datetime(df['datetime']) + timedelta(hours=5)
df['GMT'] = df['UTC'] + timedelta(hours=2)
output_file = 'common/MachineLearningModel/output/newsuperarrow.csv'
df.to_csv(output_file, index=False)
